# Keyword Baseline
- [X] Build keyword rules for all 10 categories.
- [X] Document rule sources and rationale.
- [ ] Generate matches.
- [ ] Freeze rules.
- [ ] Select thresholds.
- [ ] Run validation.
- [ ] Have another teammate review the rules/results.

# Rule Architecture

Represents the hierarchical structure of the scorecards, their evidence groups, and the rules in each evidence group.

In [11]:
from enum import Enum
from dataclasses import dataclass

class EvidenceEffect(Enum):
    """
    Fixed evidence effect on keyword baseline score
    """
    ANCHOR = 2
    SUPPORT = 1
    COUNTER = -1
    VETO = 0

@dataclass
class Rule:
    """
    A single rule within an evidence group
    Attributes:
        rule_id: Unique identifier for the rule.
        phrases: Phrases that trigger this rule.
        source: Allowed source from which the rule was derived.
        rationale: Reason the rule was accepted and how it relates to the category.
        reviewer: Person who reviewed or approved the rule, when applicable.
        behavior: Intended behavior of the rule, such as inclusion or exclusion.
    """
    rule_id: str
    phrases: list[str]
    source: str
    rationale: str
    reviewer: str
    behavior: str


@dataclass
class EvidenceGroup:
    """
    A group of related keyword rules that contributes to a category score.
    Rules within a group represent synonymous or overlapping evidence and contribute at most once per chunk.
    Attributes:
        group_id: Unique identifier for the evidence group.
        effect: Type and score contribution of the group when matched.
        rules: Keyword rules belonging to this group.
    """
    group_id: str
    effect: EvidenceEffect
    rules: list[Rule]


@dataclass
class Scorecard:
    """
    Represents a scorecard, one for each of the final 10 categories
    """
    category: str
    anchor_groups: list[EvidenceGroup]
    supporting_groups: list[EvidenceGroup]
    counterevidence_groups: list[EvidenceGroup]
    veto_groups: list[EvidenceGroup]
    threshold: int = 1

# Defining Rules

In [12]:
from utils import load_file_from_github
category_evidence = load_file_from_github("notebooks/task3_outputs/category_evidence.json")

print("Final 10 Categories:")
for category in category_evidence.keys():
    print(category)

Final 10 Categories:
Governing Law
Renewal Term
Revenue/Profit Sharing
Cap On Liability
Uncapped Liability
Termination For Convenience
Anti-Assignment
Audit Rights
License Grant
Exclusivity


In [13]:
from utils import load_split_data
import re

data = load_split_data("train")
positive_sets = data["annotation_sets"][~data["annotation_sets"].is_impossible]
positive_spans = data["spans"].merge(positive_sets[["annotation_set_id", "category_id"]], on="annotation_set_id")

def print_positive_examples(category: str):
    """Prints positive examples for a given category if it is in the final 10 categories"""
    if category in category_evidence.keys():
        for i, example in enumerate(category_evidence[category]["positive_examples"], start=1):
            print(f"{i}. {example["answer_text"]}\n")
    else:
        raise ValueError("category not in evidence")

def print_conflict_examples(categories: list[str], n: int = 5):
    """Prints positive examples for the given categories by sampling from positive_spans."""
    category_ids = [
        re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_")
        for c in categories
    ]
    matches = positive_spans[positive_spans["category_id"].isin(category_ids)]

    if matches.empty:
        print(f"No positive spans found for categories: {category_ids}")
        return

    sample = matches.sample(n=min(n, len(matches)), random_state=42)

    for i, text in enumerate(sample["answer_text"], start=1):
        print(f"{i}. {text}\n")

In [14]:
def print_negative_examples(category):
    """Prints negative examples for a given category from evidence (Task 7)."""
    for i, example in enumerate(category_evidence[category]["hard_negative_examples"], start=1):
        print(f"{i}. {example["answer_text"]}\n")

In [15]:
from utils import load_split_data

data = load_split_data("train")
positive_sets = data["annotation_sets"][~data["annotation_sets"].is_impossible]
positive_spans = data["spans"].merge(positive_sets[["annotation_set_id", "category_id"]], on="annotation_set_id")

## Governing Law
Description: Which state/country's law governs the interpretation of the contract?

Rule Summary: The rules focus on direct governed by formulations, with conflict-of-laws language as supporting evidence. Assignment language is counterevidence because it represents a different contractual category. Specific jurisdictions are avoided because jurisdiction names alone do not establish governing law.

In [16]:
print_positive_examples("Governing Law")

1. This Agreement shall be governed by and construed in accordance with the laws of the State of California without regard to its conflict of laws provisions.

2. This Agreement shall be governed by and construed in accordance with the laws of the Province of Ontario and the federal laws of Canada applicable in the Province of Ontario.

3. This Amendment shall be governed by and construed in accordance with the laws of Japan.

4. This Agreement shall be governed by and construed in accordance with the laws of New York, US, without reference to its conflict of laws principles, and shall not be governed by the United Nations Convention of International Contracts on the Sale of Goods (the Vienna Convention).

5. This Agreement will be governed by, construed and enforced in accordance with the laws of the State of Texas.



In [17]:
print_negative_examples("Governing Law")

1. Consequently either this Agreement or any of the respective rights or obligations of the Parties hereunder may be assigned or otherwise transferred, in whole or in part, in any form whatsoever (including by way of change of Control), by either Party subject to the prior written consent of the other Party, which consent shall not be unreasonably withheld or delayed, and any attempt to do so without such consent shall be null and void.

2. Either Party may, without consent of the other Party, assign this Agreement in whole to (i) in the case of RevMed, its successor in interest or assignee or purchaser, as applicable, in the case of a Change of Control or (ii) in the case of Sanofi, its successor in interest or assignee or purchaser, as applicable, in connection with the sale of all or substantially all of its assets to which this Agreement relates, or in connection with a merger, acquisition or similar transaction.

3. Neither this Agreement nor any rights or obligations hereunder sh

In [18]:
governing_law_scorecard = Scorecard(
    category="Governing Law",
    anchor_groups=[
        EvidenceGroup(
            group_id="GL-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="GL-R-001",
                    phrases=[
                        "shall be governed by and construed in accordance with the laws",
                        "will be governed by, construed and enforced in accordance with the laws",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies the law governing the agreement.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="GL-R-002",
                    phrases=[
                        "governed by the laws",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies the governing law when followed by a jurisdiction.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="GL-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="GL-R-003",
                    phrases=[
                        "without regard to its conflict of laws provisions",
                        "without reference to its conflict of laws principles",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Provides contextual support for a governing-law provision.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="GL-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="GL-R-004",
                    phrases=[
                        "assign this Agreement",
                        "assignable by a Party",
                        "assign and sublicense",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in supplied negative examples and identify assignment provisions rather than governing-law provisions.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

## Renewal Term
Description: What is the renewal term after the initial term expires? This includes automatic extensions and unilateral extensions with prior notice.

Rule Summary: The rules focus on explicit renewal periods, automatic successive renewals, and defined Renewal Terms. The conflicting examples for initial terms and mutually agreed extensions are treated as counterevidence. Specific durations and dates are avoided so the rules generalize across contracts.

In [19]:
print_positive_examples("Renewal Term")

1. Within ninety (90) days of our receipt of your notice to renew, we will furnish you with written notice of:  (i) reasons which could cause us not to grant a renewal to you including but not limited to any deficiencies which require correction and a schedule for correction by you; and (ii) our then-current requirements relating to the image, appearance, decoration, furnishing, equipping and stocking of Buffalo Wild Wings businesses, and a schedule for effecting upgrading or modifications in order to bring the Franchised Restaurant in compliance, as a condition of renewal.  Renewal of the franchise shall be conditioned upon your compliance with such requirements and continued compliance with all the terms and conditions of this Agreement up to the date of termination of the initial term.

2. This Agreement will automatically be renewed for periods of       twelve (12) months unless either Party gives six (6) months written       notice of its intent to terminate this Agreement.

3. Th

In [20]:
print_negative_examples("Renewal Term")

1. Either party may terminate this Agreement at any time with 30 days written notice.

2. ISO may terminate this Agreement prior to its expiration for cause upon prior written notice to SERVICERS as follows:

3. EHS or EHN, on the one hand, and Dr. Murray, on the other, may terminate any Services Term of this Agreement by delivering 60 days written notice to the other party.

4. ACTION FIRST ARISING, EVEN IF IT IS A CONTINUOUS ONE, OR IN THE          AGGREGATE, WITH RESPECT OF ALL CLAIMS ARISING OUT OF OR RELATED TO          THIS AGREEMENT, THE TOTAL AMOUNT ACTUALLY PAID BY CORIO UNDER THIS          AGREEMENT TO CHANGEPOINT.

5. Licensee shall have the non-exclusive right and license to develop and distribute ICE AGE 2 Wireless Products during the Term of the Agreement for all Wireless Products set forth in this Paragraph 2(c) except the Game, for which Licensee shall have the exclusive right and license to develop and distribute until December 31, 2006.



In [21]:
print_conflict_examples(["Agreement Date", "Expiration Date", "Effective Date"])

1. This Agreement will become effective as of the Effective Date and, unless sooner terminated pursuant to Sections 3.1

2. ____________

3. the 26th day of March 2020

4. The initial term of this Agreement shall be for a period of four (4) years, commencing on the Effective Date, which term may be extended for up to three (3) years by written agreement of both Parties prior to the expiration date of the initial term or any extension thereof (collectively, the "Term").

5. April 18, 2018,



In [22]:
renewal_term_scorecard = Scorecard(
    category="Renewal Term",
    anchor_groups=[
        EvidenceGroup(
            group_id="RT-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RT-R-001",
                    phrases=[
                        "automatically be renewed for periods",
                        "renewed for additional periods",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies additional periods during which the agreement is renewed.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="RT-R-002",
                    phrases=[
                        "automatic successive renewal terms",
                        "automatic extension for consecutive",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies recurring renewal or extension periods.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="RT-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RT-R-003",
                    phrases=[
                        "each such additional period, a \"Renewal Term\"",
                        "the \"Renewal Terms\"",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Explicitly designates an additional contractual period as a Renewal Term.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="RT-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="RT-R-004",
                    phrases=[
                        "may be extended for up to",
                        "by written agreement of both Parties",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in a supplied conflicting example describing an agreed extension rather than an automatic or defined renewal term.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
                Rule(
                    rule_id="RT-R-005",
                    phrases=[
                        "initial term of this Agreement",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies the initial contractual term rather than a renewal term.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

## Revenue/Profit Sharing
Description: Is one party required to share revenue or profit with the counterparty for any technology, goods, or services?

Rule Summary: The rules focus on royalty or payment language that is directly tied to revenue, sales, price, or profit. Payment terms alone are supporting evidence rather than anchors. Auditing, inspection, and renewal language is excluded as counterevidence because those provisions do not establish revenue or profit sharing.

In [23]:
print_positive_examples("Revenue/Profit Sharing")

1. In consideration for the intangible rights granted hereunder, for each Year in which the Spoken-Word Audio Sub-Section (including the Mirror Company Site) generates revenue of at [***] (the "Revenue Threshold"), Company will pay ACSI a royalty equal to [***] of all revenues generated from the Spoken-Word Audio Sub-Section (including, for the avoidance of doubt, any revenue received by Company from any Company customer who first links to the Mirror Company Site from the Spoken-Word Audio Sub-Section and who later accesses the Company Site directly) in excess of Revenue Threshold (the "Royalties") for each Year of the Term.

2. HSNS agrees to pay E.piphany an additional                   $0.005 per email for any email distributed by HSNS as a result                   of any deal it closes that either results from a lead                   generated by E.piphany or in which E.piphany assisted prior to                   closing for the first year after the deal closes.

3. During the Dis

In [24]:
print_negative_examples("Revenue/Profit Sharing")

1. This right to audit shall remain in effect throughout the life of this Agreement and for a period of three (3) years after the termination of this Agreement.

2. We have the right to inspect the proposed supplier's facilities, and require that product samples from the proposed supplier be delivered, at our option, either directly to us, or to any independent, certified laboratory that we may designate, for testing.

3. The auditor will only examine such books and records during business hours but not more than once each fiscal year while this Agreement remains in effect and for three years thereafter in order to verify expenses, Net Sales, Depomed Net Sales, PDEs or Details completed, or payments due under this Agreement.

4. At the end of such initial term, and any renewed term, as applicable, this Agreement shall automatically renew for an additional one (1) year term, unless a party provides written notice to the other parties at least six (6) months prior to the end of the

5. S

In [25]:
revenue_profit_sharing_scorecard = Scorecard(
    category="Revenue/Profit Sharing",
    anchor_groups=[
        EvidenceGroup(
            group_id="RPS-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RPS-R-001",
                    phrases=[
                        "royalty equal to",
                        "cash royalty based on",
                        "royalty payment on",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies royalty payments as part of the contractual consideration.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="RPS-R-002",
                    phrases=[
                        "of all revenues",
                        "of annual Net Sales",
                        "percentage of the invoice price",
                        "shipping profit",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly connects the payment or royalty to revenue, sales, price, or profit.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="RPS-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="RPS-R-003",
                    phrases=[
                        "pay",
                        "payment",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Provides supporting evidence for a monetary-sharing provision when combined with an anchor relating the payment to revenue, sales, price, or profit.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="RPS-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="RPS-R-004",
                    phrases=[
                        "right to audit",
                        "inspect",
                        "books and records",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in supplied negative examples and concern auditing or inspection rather than revenue or profit sharing.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="RPS-C-002",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="RPS-R-005",
                    phrases=[
                        "automatically renew",
                        "right to renew",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in supplied negative examples and concern renewal rather than revenue or profit sharing.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

## Cap On Liability
Description: Does the contract include a cap on liability upon the breach of a party’s obligation? This includes time limitation for the counterparty to bring claims or maximum amount for recovery.

Rule Summary: The rules focus on explicit maximum recovery, limits on liability, and exclusions of recoverable damages. Time-period language and exclusive-remedy language provide supporting context. Termination and assignment language is counterevidence, while explicit statements that a limitation does not apply to an obligation act as vetoes. Specific dollar amounts and party names are avoided.

In [26]:
print_positive_examples("Cap On Liability")

1. If a Buyer receives a product that fails to conform to these representations and warranties, the sole remedies of Buyer for the breach of warranty will be to: (1) reject and return the non-conforming product to Seller for a refund or credit, or a replacement conforming product, in the manner and time period provided in the SOP; (2) obtain reimbursement from Seller for actual, reasonable, substantiated out-of-pocket expenses incurred by Buyer in the recovery, return or disposal of a non-conforming product that is the subject of a mandatory product recall required under Applicable Laws or a voluntary withdrawal declared by Seller or approved by Seller (such approval not to be unreasonably withheld, conditioned or delayed); and (3) obtain indemnification from Seller for any Indemnified Claim arising from or related to the non-conforming product as provided in Section 7.

2. In addition, in the event of a material breach by Nexstar of its obligations hereunder, WYZZ shall be entitled to

In [27]:
print_negative_examples("Cap On Liability")

1. The provisions of Section 9.1 and Section 9.2 will not apply to limit the Licensee's indemnification obligations under Section 8.2, or in the case of Licensee's gross negligence or wilful misconduct.

2. THE LIMITATIONS ON LIABILITY FOR DAMAGES SET FORTH IN THIS AGREEMENT SHALL BE INAPPLICABLE TO EACH PARTY'S CONTRACTUAL OBLIGATION TO INDEMNIFY THE OTHER PARTY AS SET FORTH IN SECTIONS 2.6 AND 13.

3. TO THE MAXIMUM EXTENT PERMISSIBLE UNDER APPLICABLE LAW, EXCEPT FOR THE WILFUL MISAPPROPRIATION OR INFRINGEMENT OF THE INTELLECTUAL PROPERTY OF A PARTY TO THIS AGREEMENT, OR THE OBLIGATIONS OF THE PARTIES TO THIS AGREEMENT PURSUANT TO SECTION 13, (A) THE LIABILITY OF ANY PARTY TO THIS AGREEMENT, IF ANY, FOR DAMAGES FOR ANY CLAIM OF ANY KIND WHATSOEVER AND REGARDLESS OF THE LEGAL THEORY, WITH REGARD TO THE RIGHTS GRANTED HEREUNDER OR THE SERVICES PERFORMED HEREUNDER, SHALL NOT INCLUDE COMPENSATION, REIMBURSEMENT OR DAMAGES ON ACCOUNT OF THE LOSS OF PRESENT OR PROSPECTIVE PROFITS, EXPENDIT

In [28]:
print_positive_examples("Uncapped Liability")

1. Subject to Section 17(c), in no event shall either Party's liability under this Agreement exceed the aggregate of all amounts paid under this Agreement and amounts that have accrued but not yet been paid in the twelve (12) months preceding the event giving rise to the claim.

2. In no event will either party be liable to the other for special, incidental, or indirect damages or for any consequential damages (including lost profits or savings), even if they are informed of the possibility; provided that this Section 10.0 does not apply to Customer's failure to pay any amounts owing to IBM under this Agreement (including amounts owing for Services that would have been rendered but for Customer's breach of this Agreement).

3. With the exception of wilful misconduct by a Party, and such cases where a limitation of liability and/or indemnification is not possible under applicable law, for which cases there shall be no limitation, any and all liability and/or indemnification obligations 

In [29]:
cap_on_liability_scorecard = Scorecard(
    category="Cap On Liability",
    anchor_groups=[
        EvidenceGroup(
            group_id="COL-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="COL-R-001",
                    phrases=[
                        "up to a maximum of",
                        "shall not exceed the aggregate of all amounts",
                        "for an amount greater than the amount",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly establishes a maximum amount recoverable or payable under the agreement.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="COL-R-002",
                    phrases=[
                        "in no event will either party be liable",
                        "in no event shall any party be liable",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly introduces a contractual limitation on liability when paired with a specified damages exclusion or recovery limitation.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="COL-R-003",
                    phrases=[
                        "shall not include compensation, reimbursement or damages",
                        "will not be liable for any special, indirect, incidental or consequential damages",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly limits the types of damages recoverable for a breach or claim.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="COL-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="COL-R-004",
                    phrases=[
                        "twelve (12) months of applicable charges",
                        "in the twelve month period preceding the claim",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Provides a time-based measure for the maximum recovery or liability.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="COL-R-005",
                    phrases=[
                        "sole remedies",
                        "exclusive and in lieu of any other warranties or remedies",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Supports a limitation on the remedies available for breach when used with a specific remedy or recovery restriction.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="COL-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="COL-R-006",
                    phrases=[
                        "terminate this Agreement",
                        "advance notice thereof",
                        "assign its rights or delegate its obligations",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in supplied negative examples and identify termination or assignment provisions rather than liability caps.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[
        EvidenceGroup(
            group_id="COL-V-001",
            effect=EvidenceEffect.VETO,
            rules=[
                Rule(
                    rule_id="COL-R-007",
                    phrases=[
                        "shall not apply to limit",
                        "shall be inapplicable to",
                        "limitations on liability ... shall be inapplicable",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Explicitly removes an indemnification or other obligation from the stated liability limitation, preventing the surrounding limitation language from being treated as a cap for that obligation.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
)

## Uncapped Liability
Description: Is a party’s liability uncapped upon the breach of its obligation in the contract? This also includes uncap liability for a particular type of breach such as IP infringement or breach of confidentiality obligation.

Rule Summary: The rules focus on explicit exceptions to liability limitations, particularly language stating that a limitation does not apply or that there is no limitation for a specified breach or obligation. Indemnification and obligation references provide supporting context. Express damages exclusions and maximum-amount provisions are counterevidence, while a general liability limitation acts as a veto unless a specific uncapped exception is present.

In [30]:
print_positive_examples("Uncapped Liability")

1. Subject to Section 17(c), in no event shall either Party's liability under this Agreement exceed the aggregate of all amounts paid under this Agreement and amounts that have accrued but not yet been paid in the twelve (12) months preceding the event giving rise to the claim.

2. In no event will either party be liable to the other for special, incidental, or indirect damages or for any consequential damages (including lost profits or savings), even if they are informed of the possibility; provided that this Section 10.0 does not apply to Customer's failure to pay any amounts owing to IBM under this Agreement (including amounts owing for Services that would have been rendered but for Customer's breach of this Agreement).

3. With the exception of wilful misconduct by a Party, and such cases where a limitation of liability and/or indemnification is not possible under applicable law, for which cases there shall be no limitation, any and all liability and/or indemnification obligations 

In [31]:
print_negative_examples("Uncapped Liability")

1. THE PROVIDER SHALL NOT BE LIABLE FOR ANY INCIDENTAL OR CONSEQUENTIAL DAMAGES, INCLUDING BUT NOT LIMITED TO LOSS OF REVENUES OR LOSS OF PROFIT, ARISING OUT OF THE PROVISION OF THE SERVICES OR REPAIRS, NOTWITHSTANDING ADVANCE NOTICE FROM THE COMPANY THAT THE POSSIBILITY OF SUCH DAMAGE OR LOSS EXISTS.

2. This includes in particular the liability exclusion for lost profit, the loss of data or interruption to or errors in the operation of the Web site of the Affiliate.

3. It is expressly agreed that the Agent shall not be liable for any loss, liability, claim, damage or expense or be  required to contribute any amount pursuant to Section 9(b) or this Section 10 which in the aggregate exceeds the amount paid (excluding  reimbursable expenses) to the Agent under this Agreement.

4. This Agreement will be construed and governed in accordance with the laws of the State of Illinois, without regard to conflict of laws principles.

5. WGT reserves all rights in and to the Trademarks and all  

In [32]:
print_positive_examples("Cap On Liability")

1. If a Buyer receives a product that fails to conform to these representations and warranties, the sole remedies of Buyer for the breach of warranty will be to: (1) reject and return the non-conforming product to Seller for a refund or credit, or a replacement conforming product, in the manner and time period provided in the SOP; (2) obtain reimbursement from Seller for actual, reasonable, substantiated out-of-pocket expenses incurred by Buyer in the recovery, return or disposal of a non-conforming product that is the subject of a mandatory product recall required under Applicable Laws or a voluntary withdrawal declared by Seller or approved by Seller (such approval not to be unreasonably withheld, conditioned or delayed); and (3) obtain indemnification from Seller for any Indemnified Claim arising from or related to the non-conforming product as provided in Section 7.

2. In addition, in the event of a material breach by Nexstar of its obligations hereunder, WYZZ shall be entitled to

In [33]:
uncapped_liability_scorecard = Scorecard(
    category="Uncapped Liability",
    anchor_groups=[
        EvidenceGroup(
            group_id="UL-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="UL-R-001",
                    phrases=[
                        "does not apply to Customer's failure to pay",
                        "for which cases there shall be no limitation",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies an obligation or circumstance that is not subject to an otherwise stated liability limitation.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="UL-R-002",
                    phrases=[
                        "except in connection with a breach",
                        "except for liability arising from",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies a breach or liability category excluded from the surrounding limitation.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="UL-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="UL-R-003",
                    phrases=[
                        "subject to Section 17(c)",
                        "the obligations of the parties",
                        "indemnification obligations",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Provides contextual evidence that a particular obligation may be treated differently from the general liability limitation.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="UL-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="UL-R-004",
                    phrases=[
                        "shall not be liable for any incidental or consequential damages",
                        "shall not be liable for any loss",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases establish exclusions or limitations on liability rather than uncapped liability.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
                Rule(
                    rule_id="UL-R-005",
                    phrases=[
                        "in the aggregate exceeds the amount paid",
                        "amount greater than the amount",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly establishes a maximum amount of liability.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[
        EvidenceGroup(
            group_id="UL-V-001",
            effect=EvidenceEffect.VETO,
            rules=[
                Rule(
                    rule_id="UL-R-006",
                    phrases=[
                        "in no event will either party be liable",
                        "in no event shall either party be liable",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="A general liability limitation should prevent a broad positive unless the provision also contains a specific exception establishing uncapped liability.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
)

## Termination For Convenience
Description: Can a party terminate this contract without cause (solely by giving a notice and allowing a waiting  period to expire)?

Rule Summary: The rules focus on a unilateral right to terminate, especially when exercised at any time or for convenience through advance written notice. Unilateral and notice language provides supporting context. Renewal/non-renewal provisions and post-termination consequences are counterevidence, while notice tied specifically to the end of an initial or renewal term is treated as a veto against a termination-for-convenience match.

In [34]:
print_positive_examples("Termination For Convenience")

1. Written notice of intention to withdraw must be served in writing upon the remaining Participants at least Thirty (30) business days prior to the withdrawal date.

2. The Agreement rests, for all that, cancellable at any time by any of the parties before the expiry date of the Agreement or any of itsrenewals, upon three months prior written notice.

3. Notwithstanding the foregoing, either party may terminate this Agreement at any time without liability by providing one hundred eighty (180) days written notice to the other party.

4. Party A is entitled to unilaterally terminate this Agreement within three natural months from the signing date of this Agreement.

5. Licensee may terminate this Agreement for convenience upon eighteen (18) months' advance written notice to Bioeq; provided, however, that any such termination for convenience shall not become effective prior to twelve (12) months after the First Commercial Sale of the first Licensed Product.



In [35]:
print_negative_examples("Termination For Convenience")

1. MBE shall have the right to elect by written notice to the Company at any time between two (2) and six (6) months prior to the end of such initial term or any subsequent Renewal Period (as defined below), to notify the Company that MBE elects to seek to extend such term for additional two (2)-year periods (each a "Renewal Period") In the event of such election, MBE and the Company shall have  -------------- a period of sixty (60) days in which to negotiate commercially reasonable Basic Fees, Bounty Fees and eBay Fees (and other applicable fees) ("Fee Schedule")                                                               ------------ under which the Company would be willing to renew this Agreement for such Renewal Period.

2. Upon expiry of the Initial Term, this Agreement [*****] unless a Notice of non-renewal is given by either Party to the other Party [*****] prior to the expiry of the Initial Term or the end of a renewal period, if any.

3. This Agreement shall commence on the 

In [36]:
print_conflict_examples(["Notice Period to Terminate Renewal", "Post-Termination Services"])

1. D.   You will take such action as may be necessary to cancel or assign to us or our designee, at our option, any assumed name rights or equivalent registration filed with state, city, or county authorities which contains the name "Buffalo Wild Wings," "bw-3" or any Mark, and you will furnish us with evidence satisfactory to us of compliance with this obligation within thirty (30) days after termination or expiration of this Agreement.

2. After the Initial Term (including any extension thereto made in accordance with the preceding sentence), the Agreement may be extended on a yearly basis up to ten (10) years at Vericel's sole discretion, with renewal notice to be provided to MediWound no later than twelve (12) months prior to the expiry of any yearly extension (the "Renewal Term", and the Initial Term, together with the Renewal Term, if any, the "Term"); provided that unless otherwise agreed by the Parties, the Term of this Agreement (including the Initial Term, any extension of th

In [37]:
termination_for_convenience_scorecard = Scorecard(
    category="Termination For Convenience",
    anchor_groups=[
        EvidenceGroup(
            group_id="TFC-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="TFC-R-001",
                    phrases=[
                        "cancellable at any time by any of the parties",
                        "may terminate this Agreement at any time without liability",
                        "terminate this Agreement for convenience",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly establishes a party's ability to terminate without requiring a stated cause.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="TFC-R-002",
                    phrases=[
                        "prior written notice",
                        "advance written notice",
                        "providing ... days written notice",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies the notice mechanism through which a termination right is exercised.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="TFC-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="TFC-R-003",
                    phrases=[
                        "unilaterally terminate",
                        "at any time",
                        "upon",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Supports a unilateral termination right when used with termination and notice language.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="TFC-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="TFC-R-004",
                    phrases=[
                        "renewal period",
                        "renew this Agreement",
                        "automatically be renewed for periods",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases concern renewal or non-renewal rather than a right to terminate the agreement for convenience.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
                Rule(
                    rule_id="TFC-R-005",
                    phrases=[
                        "after termination",
                        "employment ... is terminated",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases describe consequences of termination rather than a contractual termination-for-convenience right.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[
        EvidenceGroup(
            group_id="TFC-V-001",
            effect=EvidenceEffect.VETO,
            rules=[
                Rule(
                    rule_id="TFC-R-006",
                    phrases=[
                        "prior to the end of such initial term",
                        "prior to the expiry of the Initial Term",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies notice concerning renewal or non-renewal at the end of a term rather than termination for convenience during the term.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
)

## Anti-Assignment
Description: Is consent or notice required of a party if the contract is assigned to a third party?

Rule Summary: The rules focus on assignment provisions that explicitly require prior written consent or address assignment with notice. Assignment and transfer language provides supporting context. Governing-law and termination provisions are counterevidence, while standalone assignments of intellectual property or other rights are vetoed because they do not establish an anti-assignment restriction.

In [38]:
print_positive_examples("Anti-Assignment")

1. Consequently either this Agreement or any of the respective rights or obligations of the Parties hereunder may be assigned or otherwise transferred, in whole or in part, in any form whatsoever (including by way of change of Control), by either Party subject to the prior written consent of the other Party, which consent shall not be unreasonably withheld or delayed, and any attempt to do so without such consent shall be null and void.

2. Either Party may, without consent of the other Party, assign this Agreement in whole to (i) in the case of RevMed, its successor in interest or assignee or purchaser, as applicable, in the case of a Change of Control or (ii) in the case of Sanofi, its successor in interest or assignee or purchaser, as applicable, in connection with the sale of all or substantially all of its assets to which this Agreement relates, or in connection with a merger, acquisition or similar transaction.

3. Neither this Agreement nor any rights or obligations hereunder sh

In [39]:
print_negative_examples("Anti-Assignment")

1. The construction, interpretation and performance of this Agreement and all transactions under it shall be governed by the law of the State of Israel, without giving effect to choice of law rules, and both Parties consent to jurisdiction by the courts of the City of Haifa.

2. This Agreement shall be governed by and construed in accordance with the laws of the Province of British Columbia and the federal laws of Canada applicable therein, excluding its conflict-of-laws rules.

3. This Agreement shall be governed by and construed under the laws of Hong Kong without regard to choice of laws principles.

4. The Company will have the ability to terminate this Agreement by giving 60 days' prior  written notice to the Contractor.

5. Any books of accounts or records shall not be inspected more than once.



In [40]:
print_conflict_examples(["Change of Control", "IP Ownership Assignment"])

1. Other than as permitted under Section 8.5.2 (Use of Collaboration Data), in no event shall BLI, and BLI shall cause its Affiliates to not, file any patent applications covering (or support existing patent applications covering) [***] and, in the event that BLI (or its Affiliates) do file one or more of such patent applications, BLI will and hereby does assign, and shall cause its employees, agents and contractors to assign, to Ginkgo all rights, title and interests in, to and under such patent applications.

2. In case [Party] has made or contributed to any invention forming part of the  Results, [Party] shall promptly inform [CERES/IGER] thereof in writing and shall assign its rights and cause its employees and staff  members to assign their rights in any such invention to [CERES/IGER].

3. if such rights comprise (i) intellectual property that constitutes predominantly communication software or related communication hardware or other technology, including without limitation, any u

In [41]:
anti_assignment_scorecard = Scorecard(
    category="Anti-Assignment",
    anchor_groups=[
        EvidenceGroup(
            group_id="AA-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="AA-R-001",
                    phrases=[
                        "may be assigned ... subject to the prior written consent",
                        "assignable by a Party without the prior written consent",
                        "may not be assigned ... without the prior written consent",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly establishes that assignment is subject to the other party's prior written consent.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="AA-R-002",
                    phrases=[
                        "on notice to but without the other Party's consent",
                        "without consent of the other Party",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly addresses whether assignment requires consent or notice.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="AA-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="AA-R-003",
                    phrases=[
                        "assign or otherwise transfer",
                        "assignable by a Party",
                        "transfer or otherwise deal with the rights",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies assignment or transfer language that supports an assignment restriction when paired with consent or notice requirements.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="AA-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="AA-R-004",
                    phrases=[
                        "governed by and construed in accordance with the laws",
                        "governed by and construed under the laws",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in supplied negative examples and identify governing-law provisions rather than assignment restrictions.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
                Rule(
                    rule_id="AA-R-005",
                    phrases=[
                        "terminate this Agreement by giving",
                        "prior written notice to the Contractor",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases concern termination notice rather than consent or notice requirements for assignment.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[
        EvidenceGroup(
            group_id="AA-V-001",
            effect=EvidenceEffect.VETO,
            rules=[
                Rule(
                    rule_id="AA-R-006",
                    phrases=[
                        "assign its rights",
                        "assign their rights",
                        "hereby irrevocably assigns",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These supplied conflict examples concern assignment of intellectual property or other rights as part of ownership transfer, rather than restrictions requiring consent or notice for assignment of the contract.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
)

## Audit Rights
Description: Does a party have the right to  audit the books, records, or physical locations of the counterparty to ensure compliance with the contract?

Rule Summary: The rules focus on explicit rights to audit, inspect, examine, copy, or access books, records, or properties. Notice, business-hours, and record-access language provides supporting context. Royalty, revenue-sharing, fee, and licensing language is counterevidence because it represents other contractual provisions.

In [42]:
print_positive_examples("Audit Rights")

1. The Auditing Party may cause the Auditor to perform such an audit not more than once in any 12-month period, unless a prior audit within the past two years revealed that the amount owed by the Audited Party to the Auditing Party was underpaid in excess of 8% of the amount owed, in which case an audit may be performed no more frequently than twice in any 12-month period.

2. Franchisee shall fully cooperate with Pretzel Time's representatives and independent accountants hired by Pretzel Time to conduct any such inspection or audit.

3. Premier, shall have the right, directly or through its representative, to inspect, copy, and audit all such records upon reasonable request and during normal business hours, acknowledging that access to accounting and purchasing records will be limited to those supporting pass-through materials costs and purchases of Premier specified equipment if any.

4. SHPS shall have the right, upon reasonable prior written notice, to examine, copy and audit such 

In [43]:
print_negative_examples("Audit Rights")

1. In consideration of the License granted and other good and valuable consideration provided by T&B to LEA, LEA shall pay to T&B a base royalty ("Base Royalty") in the amount of [●%] of LEA's monthly Cash Sales for Cash Sales of up to [$●].

2. The revenue share for the Program is stated in Exhibit B.

3. License Renewal and Maintenance Fees will be split evenIy with 50% distributed to Developer and 50% distributed to Distributor, also subject to paragraph 4 of this section 2.3.

4. As part of the exclusive distribution right granted in this Section 2, Vendor hereby grants Distributor the non- exclusive, non-transferable right to use and display Vendor's trademarks, logos, Product photographs and images, Product advertising and promotional copy, including but not limited to the materials contained in Vendor's website, in connection with the promotion, advertising and distribution of the Products.

5. A majority of the Restaurant owners in the Regional Fund may vote to increase the amo

In [44]:
audit_rights_scorecard = Scorecard(
    category="Audit Rights",
    anchor_groups=[
        EvidenceGroup(
            group_id="AR-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="AR-R-001",
                    phrases=[
                        "perform such an audit",
                        "conduct any such inspection or audit",
                        "inspect, copy, and audit",
                        "examine, copy and audit such records",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly establishes a contractual right to audit or inspect records.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="AR-R-002",
                    phrases=[
                        "full access to all properties, books of account, and records",
                        "right ... to inspect",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly establishes access to records or physical locations for inspection.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="AR-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="AR-R-003",
                    phrases=[
                        "upon reasonable request",
                        "upon reasonable prior written notice",
                        "normal business hours",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Provides contextual support for an audit or inspection right when paired with direct audit language.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="AR-R-004",
                    phrases=[
                        "books of account",
                        "accounting and purchasing records",
                        "such records",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies the records subject to the audit or inspection right.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="AR-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="AR-R-005",
                    phrases=[
                        "base royalty",
                        "revenue share",
                        "maintenance fees",
                        "Gross Revenues",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in supplied negative examples and concern payments, revenue sharing, or fees rather than audit rights.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
                Rule(
                    rule_id="AR-R-006",
                    phrases=[
                        "non-transferable right to use and display",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies a trademark or licensing right rather than an audit right.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

The positive examples consistently establish a contractual right to audit, inspect, examine, copy, or access records, with some provisions also specifying notice, frequency, location, business hours, or auditor requirements. The negative examples concern royalties, revenue sharing, license rights, and financial contributions without establishing an audit right. The scorecard therefore treats procedural terms such as notice or frequency as supporting evidence only when connected to an audit or record-inspection context, rather than as independent audit indicators.

## License Grant
Description: Does the contract contain a license granted by one party to its counterparty?

Rule Summary: The rules focus on explicit grant formulations such as "hereby grants to" and "right and license to use". Permitted uses and sublicensing language provide supporting evidence. Assignment and transfer of ownership rights are counterevidence because they represent a different type of rights transaction. The rules avoid using "license" alone because it can occur in provisions that do not establish the relevant grant.

In [45]:
print_positive_examples("License Grant")

1. Subject to the terms and conditions hereof, drkoop.com hereby represents that it has the power and authority to grant, and does hereby grant to Sponsor a non-exclusive, non-transferable, royalty-free, worldwide license to reproduce and display all logos, trademarks, trade names and similar identifying material relating to drkoop.com and, solely as allowed pursuant to this Agreement, to the Dr. C. Everett Koop name (collectively, the "drkoop.com Marks") solely in connection with the promotion, marketing and distribution of the parties and the Sites in accordance with the terms hereof, provided, however, that Sponsor shall, other than as specifically provided for in Section 4.4 of this Agreement, not make any specific use of any drkoop.com Marks without first submitting a sample of such use to drkoop.com and obtaining its prior consent, which consent shall not be unreasonably withheld.

2. During the term of this Agreement, and subject to the terms and conditions hereof, STAAR hereby 

In [46]:
print_negative_examples("License Grant")

1. MediWound shall have an exclusive license under patents and other intellectual property, to develop, use, manufacture, market and sell the Product for burn treatment in humans;

2. Subject to the provisions of this Agreement, Bioeq hereby grants to Licensee an exclusive (even as to Bioeq), milestone- and royalty-bearing, non-transferable license (including the right to grant sublicenses only to the extent permitted by Section 2.1.2) under the Licensed Technology (including the Licensed Patents) to use, sell, have sold, import, have imported or otherwise Commercialize the Licensed Products in the Field in the Territory.

3. Based on the foregoing, and subject to the terms and conditions contained in this Agreement, the Agents severally and not jointly agree to act as, and the Corporation appoints the Agents as, the exclusive agents of the Corporation to offer the Offered Shares for sale on the Closing Date (as defined herein) in the Selling Jurisdictions (as defined herein) on a priv

In [47]:
print_conflict_examples(["IP Ownership Assignment", "Assignment of Contract"])

1. Ono shall assign (or cause to be assigned) to Array or its designee, at Array's cost, except in case of termination by Array pursuant to Section 13.3 or 13.4 or by Ono pursuant to Section 13.2, in which case the expenses will be borne by Ono, (or to the extent not so assignable, Ono shall take all reasonable actions to make available to Array or its designee the benefits of) all Regulatory Filings for the Product in the Ono Territory, including any such Regulatory Filings made or owned by its Affiliates and/or Sublicensees.

2. In the event LEA shall be deemed to have acquired any ownership rights in the Licensed Intellectual Property, the LEA shall assign, and agrees to execute all documents reasonably requested by T&B to assign, all such rights in the Licensed Intellectual Property to T&B or its nominee.

3. (e) transfer to Xencor all of Aimmune's right, title and interest in and to any and all Development-related data and Commercialization Data Controlled by Aimmune for the Produ

In [48]:
license_grant_scorecard = Scorecard(
    category="License Grant",
    anchor_groups=[
        EvidenceGroup(
            group_id="LG-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="LG-R-001",
                    phrases=[
                        "hereby grants to",
                        "does hereby grant to",
                        "agrees and grants to",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies one party granting rights to the counterparty.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="LG-R-002",
                    phrases=[
                        "right and license to use",
                        "non-exclusive ... license to",
                        "non-exclusive ... right to use and display",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly identifies a granted license or right of use.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="LG-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="LG-R-003",
                    phrases=[
                        "reproduce and display",
                        "use the trade name, trademarks, and logos",
                        "use and display",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies the permitted uses associated with a granted license or right.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="LG-R-004",
                    phrases=[
                        "without the right to grant sublicenses",
                        "right to grant sublicenses",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Provides contextual evidence of a license provision by addressing sublicensing rights.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="LG-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="LG-R-005",
                    phrases=[
                        "assign all such rights",
                        "assign ... all rights, title and interest",
                        "transfer ... right, title and interest",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases concern assignment or transfer of ownership rights rather than granting a license to use rights.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
                Rule(
                    rule_id="LG-R-006",
                    phrases=[
                        "Regulatory Filings",
                        "Development-related data and Commercialization Data",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases occur in supplied conflicting examples concerning transfer of filings, data, or assets rather than a license grant.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

The positive examples consistently establish a grant of a right or license to use identified intellectual property, trademarks, materials, or business-related know-how. The negative examples include explicit patent and technology license grants, demonstrating that the presence of the word “license” alone is insufficient to distinguish this category. The scorecard therefore relies on the specific license-grant patterns represented by the positive examples and requires the surrounding licensed subject and contractual context rather than treating generic licensing terminology as sufficient evidence.

## Exclusivity
Description: Is there an exclusive dealing  commitment with the counterparty? This includes a commitment to procure all “requirements” from one party of certain technology, goods, or services or a prohibition on licensing or selling technology, goods or services to third parties, or a prohibition on  collaborating or working with other parties), whether during the contract or  after the contract ends (or both).

Rule Summary: The rules focus on restrictions against dealing with other companies or competitors and explicit exclusive rights to sell or provide products or services. References to competitors and the contract term provide supporting context. Exclusive licensing, inspection, and renewal provisions are treated as counterevidence because they do not establish exclusive dealing.

In [49]:
print_positive_examples("Exclusivity")

1. Reseller shall not obtain the           TouchStar Software or Support Services (or any software or services           which compete with the TouchStar Software) for sale from any Entity           other than TouchStar or its authorized agents.

2. Nantz Communications and Nantz expressly agree that the Endorsement will not be granted to anyone other than the Company for use during the Term in connection with the advertisement and promotion of sportswear apparel, hats and shoes.

3. During the Term of this Agreement, except as otherwise permitted by this Section 3(a)(v), VS agrees that it shall not enter into the same or substantially similar Commitments with any other company or entity which performs clinical research services the same or similar to those provided by PPD or any PPD affiliate (collectively, "PPD Competitor"), nor shall VS provide preferred pricing to a PPD Competitor which is better than that provided by VS hereunder to PPD.

4. The Company hereby appoints and grants 

In [50]:
print_negative_examples("Exclusivity")

1. Without limiting the foregoing, Magenta may use and disclose Bachem Intellectual Property to the extent necessary in connection with the prosecution, maintenance and enforcement of Magenta Developed Intellectual Property.

2. Subject to the terms and conditions of this Agreement, Licensor hereby grants to Licensee, and Licensee hereby accepts from Licensor, an exclusive, non-transferable (except as set forth in Section 10.7) and non-sublicensable (except as provided in Section 2.1(c)) license to use the Licensed Content in connection with websites associated with the Licensed Domain Names until the earlier of (i) termination or expiration of this Agreement, or (ii) termination or expiration of the Agency Agreement, provided, however, that in the event the Agency Agreement is amended or restated, such amendment or restatement shall not be deemed a termination or expiration of the Agency Agreement.

3. shall retain those licenses granted to it under this Agreement, provided that the l

In [51]:
exclusivity_scorecard = Scorecard(
    category="Exclusivity",
    anchor_groups=[
        EvidenceGroup(
            group_id="EX-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="EX-R-001",
                    phrases=[
                        "shall not obtain",
                        "will not be granted to anyone other than",
                        "shall not enter into the same or substantially similar Commitments with any other",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly restricts a party from obtaining, granting, or entering into comparable arrangements with third parties.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="EX-R-002",
                    phrases=[
                        "exclusive right to sell",
                        "exclusivity rights granted",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Directly establishes an exclusive commercial dealing right.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="EX-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="EX-R-003",
                    phrases=[
                        "other than TouchStar or its authorized agents",
                        "any other company or entity",
                        "PPD Competitor",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Identifies the third parties or competitors from whom the restricted goods, services, or commitments cannot be obtained.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
                Rule(
                    rule_id="EX-R-004",
                    phrases=[
                        "during the Term",
                        "continuing each year of the Term thereafter",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="Provides temporal context for an exclusivity commitment.",
                    reviewer="Not applicable",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="EX-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="EX-R-005",
                    phrases=[
                        "exclusive ... license to use",
                        "retain those licenses granted to it",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases concern exclusive licensing rights rather than an exclusive dealing commitment with a counterparty.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
                Rule(
                    rule_id="EX-R-006",
                    phrases=[
                        "right of entry and inspection",
                        "option ... to extend the Term",
                    ],
                    source="annotated Span text, nearby text, and manually reviewed hard negatives from the model-training portion of the official training Contracts",
                    rationale="These phrases concern inspection or renewal rights rather than exclusivity.",
                    reviewer="Not applicable",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

# Scoring

Confirm that it covers all 10 categories and that every rule and evidence group has a unique ID.

In [194]:
scorecards = {
    "Governing Law" : governing_law_scorecard,
    "Renewal Term" : renewal_term_scorecard,
    "Revenue/Profit Sharing" : revenue_profit_sharing_scorecard,
    "Cap On Liability" : cap_on_liability_scorecard,
    "Uncapped Liability" : uncapped_liability_scorecard,
    "Termination For Convenience" : termination_for_convenience_scorecard,
    "Anti-Assignment" : anti_assignment_scorecard,
    "Audit Rights" : audit_rights_scorecard,
    "License Grant" : license_grant_scorecard,
    "Exclusivity" : exclusivity_scorecard
}

all_rule_ids = []
all_group_ids = []

for category, scorecard in scorecards.items():
    groups = (
        scorecard.anchor_groups
        + scorecard.supporting_groups
        + scorecard.counterevidence_groups
        + scorecard.veto_groups
    )

    for group in groups:
        all_group_ids.append(group.group_id)

        for rule in group.rules:
            all_rule_ids.append(rule.rule_id)

print("Categories:", len(scorecards))
print("Unique categories:", len(set(scorecards)))

print("Rules:", len(all_rule_ids))
print("Unique rule IDs:", len(set(all_rule_ids)))

print("Groups:", len(all_group_ids))
print("Unique group IDs:", len(set(all_group_ids)))

assert len(scorecards) == 10
assert len(all_rule_ids) == len(set(all_rule_ids))
assert len(all_group_ids) == len(set(all_group_ids))

Categories: 10
Unique categories: 10
Rules: 57
Unique rule IDs: 57
Groups: 35
Unique group IDs: 35


Freeze the scorecards in a `.pkl` file.

In [176]:
import pickle
with open("task11_outputs/frozen_scorecards.pkl", "wb") as f:
    pickle.dump(scorecards, f)

## Import Data & Make Chunks
Using the validation data, we chunk the contracts by paragraphs (deemed the best chunking strategy). As a result, we have 2,873 chunks and 28,730 chunk-category prediction rows.

Configuration:
- DistilRoBERTa tokenizer
- max 512 tokens
- 64-token overlap for long paragraphs

In [140]:
from chunking_helper import create_paragraph_chunks
from utils import load_split_data
# Import the Hugging Face tokenizer used to create model-ready token chunks.
# A fast tokenizer is required because it can return character offsets.
from transformers import AutoTokenizer

TOKENIZER_NAME = "distilroberta-base"

# Load the tokenizer.
# TODO: add option for token... warning is annoying
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME,
    use_fast=True
)

print("Tokenizer:", TOKENIZER_NAME)
print("Model maximum length:", tokenizer.model_max_length)

validation_data = load_split_data("validation")
chunks = create_paragraph_chunks(
    documents_df=validation_data["documents"],
    tokenizer=tokenizer,
    chunk_size=512,
    long_paragraph_overlap=64,
    method_name="paragraph_based"
)

print("Number of Chunks: ", len(chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (636 > 512). Running this sequence through the model will result in indexing errors


Tokenizer: distilroberta-base
Model maximum length: 512
Number of Chunks:  2873


`create_paragraph_chunks` gives us the following columns:
- method
- contract_id
- context_group_id
- chunk_id
- chunk_number
- char_start
- char_end
- num_tokens

We append the chunk text:

In [141]:
chunks["text"] = chunks.apply(
    lambda row: validation_data["documents"].loc[
        validation_data["documents"]["contract_id"] == row["contract_id"], "context"
    ].iloc[0][row["char_start"]:row["char_end"]],
    axis=1
)

chunks[["contract_id", "chunk_id", "text"]].head()

,contract_id,chunk_id,text
0,contract_0013,contract_0013__chunk_0000,DISTRIBUTION AND SERVICES AGREEMENT January 18...
1,contract_0013,contract_0013__chunk_0001,such orders and to make such sales. Section 1....
2,contract_0013,contract_0013__chunk_0002,"Source: INTEGRITY FUNDS, 485BPOS, 1/21/2020\n\..."
3,contract_0013,contract_0013__chunk_0003,"(f) sales commissions and other fees paid, tog..."
4,contract_0013,contract_0013__chunk_0004,ENTATIONS AND WARRANTIES Section 3.1 The Fund ...


## Get Results

Use frozen the scorecards and the classifier from `keyword_classifier.py`.

In [179]:
from keyword_classifier import KeywordClassifier

with open("task11_outputs/frozen_scorecards.pkl", "rb") as f:
    frozen_scorecards = pickle.load(f)

kc = KeywordClassifier(frozen_scorecards)
results = kc.score(chunks)
results.head()

,contract_id,chunk_id,category_name,score,evidence
0,contract_0013,contract_0013__chunk_0000,Governing Law,0,[]
1,contract_0013,contract_0013__chunk_0000,Renewal Term,0,[]
2,contract_0013,contract_0013__chunk_0000,Revenue/Profit Sharing,0,[]
3,contract_0013,contract_0013__chunk_0000,Cap On Liability,0,[]
4,contract_0013,contract_0013__chunk_0000,Uncapped Liability,0,[]


Confirm that the classifier covers all 10 categories.

In [146]:
num_chunks = len(chunks)
all_categories_represented = (results.groupby("category_name").agg(num_chunks=('chunk_id', 'nunique'))["num_chunks"] == num_chunks).all()
print("All 10 categories represented:", all_categories_represented)

All categories represented: True


## Evaluate

Create the gold labels (true chunk categories) for evaluation.

In [183]:
import pandas as pd

def create_gold_labels(chunks, validation_data, target_categories):
    spans = validation_data["spans"].merge(
        validation_data["annotation_sets"],
        on="annotation_set_id"
    )

    categories = validation_data["categories"]
    target_category_keys = categories[
        categories["category_name"].isin(target_categories)
    ][["category_id", "category_name"]]

    spans = spans[
        spans["category_id"].isin(target_category_keys["category_id"])
    ]

    gold_labels = []

    for _, chunk in chunks.iterrows():
        for _, category in target_category_keys.iterrows():
            matches = (
                (spans["contract_id"] == chunk["contract_id"]) &
                (spans["category_id"] == category["category_id"]) &
                (spans["answer_start"] < chunk["char_end"]) &
                (spans["answer_end"] > chunk["char_start"])
            )

            gold_labels.append({
                "chunk_id": chunk["chunk_id"],
                "category_id": category["category_id"],
                "category_name": category["category_name"],
                "gold": int(matches.any())
            })

    return pd.DataFrame(gold_labels)

gold_labels = create_gold_labels(
    chunks,
    validation_data,
    frozen_scorecards.keys()
)
gold_labels.head()

,chunk_id,category_id,category_name,gold
0,contract_0013__chunk_0000,renewal_term,Renewal Term,0
1,contract_0013__chunk_0000,governing_law,Governing Law,0
2,contract_0013__chunk_0000,exclusivity,Exclusivity,0
3,contract_0013__chunk_0000,termination_for_convenience,Termination For Convenience,0
4,contract_0013__chunk_0000,anti_assignment,Anti-Assignment,0


Create the evaluation set containing the chunks and their respective category name.

In [185]:
evaluation = results.merge(
    gold_labels,
    on=["chunk_id", "category_name"],
    how="left"
)

selected_thresholds, threshold_results = kc.select_thresholds(
    evaluation
)

selected_thresholds

,category_name,threshold,f1,recall
0,Anti-Assignment,1,0.044444,0.023256
1,Audit Rights,1,0.541935,0.552632
2,Cap On Liability,1,0.134831,0.083333
3,Exclusivity,2,0.080000,0.041667
4,Governing Law,1,0.468468,0.305882
5,License Grant,1,0.510638,0.363636
6,Renewal Term,1,0.058824,0.030303
7,Revenue/Profit Sharing,2,0.120000,0.065217
8,Termination For Convenience,3,0.224719,0.285714
9,Uncapped Liability,1,0.368421,0.304348


## Weak/Unsuccessful Behavior

This Keyword Baseline shows an overall weak performance across most categories. Anti-Assignment, Renewal Term, Exclusivity, and Reevnue/Profit Sharing had particularly low recall and F1 scores, indicating that the keyword rules frequently failed to detect annotated instances. Audit Rights and License Grant had the highest F1 scores but still missed a substantial portion of annotated instances. These results indicate the limitations of a deterministic keyword approach, including missed linguistic variations and context-dependent cases. They would require more in-depth rules backed by  expertise from industry professionals.

In [189]:
# save the thresholds
selected_thresholds.to_csv("task11_outputs/selected_thresholds_v1.csv")
threshold_results.to_csv("task11_outputs/threshold_results_v1.csv")

# Validation Results


In [187]:
predictions = kc.predict(scores, selected_thresholds)
print(len(predictions) == 28730)
print(predictions["prediction"].isin([0, 1]).all())

True
True


Verify the following:
- 2,873 chunks
- 10 categories
- one unique prediction per chunk-category pair
- 28,730 total predictions
- predictions are 0/1
- scorer output has no gold labels

In [197]:
assert len(predictions) == 28730
assert predictions[["chunk_id", "category_name"]].duplicated().sum() == 0
assert predictions["chunk_id"].nunique() == 2873
assert predictions["category_name"].nunique() == 10
assert predictions["prediction"].isin([0, 1]).all()
assert "gold" not in scores.columns

print("Validation coverage: PASS")

Validation coverage: PASS


In [193]:
validation_results = selected_thresholds[
    ["category_name", "threshold", "f1", "recall"]
].copy()

gold_counts = (
    evaluation.groupby("category_name")["gold"]
    .sum()
    .rename("gold_positives")
)

predicted_counts = (
    predictions.groupby("category_name")["prediction"]
    .sum()
    .rename("predicted_positives")
)

validation_results = (
    validation_results
    .merge(gold_counts, on="category_name")
    .merge(predicted_counts, on="category_name")
)

validation_results = validation_results[
    [
        "category_name",
        "threshold",
        "f1",
        "recall",
        "gold_positives",
        "predicted_positives"
    ]
]

validation_results.to_csv("task11_outputs/validation_results_v1.csv")
validation_results

,category_name,threshold,f1,recall,gold_positives,predicted_positives
0,Anti-Assignment,1,0.044444,0.023256,86,4
1,Audit Rights,1,0.541935,0.552632,76,79
2,Cap On Liability,1,0.134831,0.083333,72,17
3,Exclusivity,2,0.080000,0.041667,48,2
4,Governing Law,1,0.468468,0.305882,85,26
5,License Grant,1,0.510638,0.363636,99,42
6,Renewal Term,1,0.058824,0.030303,33,1
7,Revenue/Profit Sharing,2,0.120000,0.065217,46,4
8,Termination For Convenience,3,0.224719,0.285714,35,54
9,Uncapped Liability,1,0.368421,0.304348,23,15
